# Structured HK epsilon robustness sweep

Abundant-data diagnostic for the structured Hegselmann-Krause environments. Runs the bounded and unbounded alpha identifiers separately while fixing the final simplified controller to `lambda_mix=1.0`.

The only environment parameter swept is the HK confidence radius `epsilon = [0.15, 0.20, 0.25, 0.275, 0.30, 0.325, 0.35, 0.40, 0.50]`. The three structured HK families, five environment seeds, training settings, budgets, and all other environment parameters are unchanged.


In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

HERE = Path.cwd()
BASE_NOTEBOOK = HERE / "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed.ipynb"
if not BASE_NOTEBOOK.exists():
    raise FileNotFoundError(f"Missing committed abundant-data base notebook: {BASE_NOTEBOOK}")

VARIANT = os.environ.get("HK_EPS_VARIANT", "").strip().lower()
if VARIANT not in {"bounded", "unbounded"}:
    raise RuntimeError(
        "Set HK_EPS_VARIANT to 'bounded' or 'unbounded'. "
        "Use the supplied launchers rather than running worker mode manually."
    )

STUDY_NAME_OVERRIDE = "hk_eps_b" if VARIANT == "bounded" else "hk_eps_u"
PIPELINE_OVERRIDE = f"2026-09-16-hk-eps-{VARIANT}-lambda1-v1"

print("HK epsilon robustness base:", BASE_NOTEBOOK.name)
print("Identifier variant:", VARIANT)
print("Study namespace:", STUDY_NAME_OVERRIDE)
print("Control matrix: lambda_mix = 1.0")

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))
patched_identifier = (VARIANT == "bounded")
patched_study = False
patched_results = False
patched_lambda = False
patched_env_registry = False
patched_common_row = False
patched_config = False
patched_cells = []

CUSTOM_ENV_CELL = '# Diagnostic scope: structured Hegselmann-Krause only.\n# Keep the three final structured families and five environment seeds, and\n# sweep only the HK confidence radius.\nHK_EPS_VALUES = [0.15, 0.20, 0.25, 0.275, 0.30, 0.325, 0.35, 0.40, 0.50]\n\nENVIRONMENT_SPECS: list[dict[str, Any]] = []\n\nfor eps in HK_EPS_VALUES:\n    eps_tag = f"{int(round(1000 * float(eps))):03d}"\n    for family in HK_FAMILIES:\n        for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:\n            raw, A = build_hk_graph(family, environment_seed)\n            x0 = build_hk_x0(environment_seed)\n\n            edge_mask = np.asarray(A, dtype=float) > 0\n            pair_distance = np.abs(x0[:, None] - x0[None, :])\n            active_edge_fraction = float(\n                np.sum(edge_mask & (pair_distance <= float(eps))) / np.sum(edge_mask)\n            )\n            bridge_gap = float(abs(x0[HK_LOW_HUB] - x0[HK_HIGH_HUB]))\n\n            ENVIRONMENT_SPECS.append({\n                "environment_index": len(ENVIRONMENT_SPECS),\n                "environment_id": (\n                    f"hk__eps{eps_tag}__{family}__env{int(environment_seed):02d}"\n                ),\n                "scenario_class": "structured_mechanism",\n                "family": family,\n                "dynamics": "hegselmannkrause",\n                "topology_seed": None,\n                "opinion_seed": None,\n                "environment_seed": int(environment_seed),\n                "dynamics_params": {\n                    "hk_epsilon": float(eps),\n                    "hk_include_self": True,\n                },\n                "hk_epsilon": float(eps),\n                "initial_active_edge_fraction": active_edge_fraction,\n                "initial_bridge_opinion_gap": bridge_gap,\n                "initial_bridge_active": bool(bridge_gap <= float(eps)),\n                "raw": raw,\n                "A": A,\n                "x0": x0,\n                "v_true": centrality_from_A(A),\n            })\n\nexpected_envs = len(HK_EPS_VALUES) * len(HK_FAMILIES) * len(STRUCTURED_ENVIRONMENT_SEEDS)\nassert len(ENVIRONMENT_SPECS) == expected_envs\n\nregistry_df = pd.DataFrame([\n    {\n        "environment_index": spec["environment_index"],\n        "environment_id": spec["environment_id"],\n        "scenario_class": spec["scenario_class"],\n        "family": spec["family"],\n        "dynamics": spec["dynamics"],\n        "environment_seed": spec["environment_seed"],\n        "hk_epsilon": spec["hk_epsilon"],\n        "initial_active_edge_fraction": spec["initial_active_edge_fraction"],\n        "initial_bridge_opinion_gap": spec["initial_bridge_opinion_gap"],\n        "initial_bridge_active": spec["initial_bridge_active"],\n    }\n    for spec in ENVIRONMENT_SPECS\n])\n\nprint(\n    registry_df.groupby(["hk_epsilon", "family"])\n    .size()\n    .rename("n_env")\n    .to_string()\n)\nprint("Distinct environments:", len(ENVIRONMENT_SPECS))\nprint("Learned evaluations:", len(ENVIRONMENT_SPECS) * len(LEARNING_SEEDS))\nprint(\n    "\\nInitial bridge-active rate by epsilon:\\n",\n    registry_df.groupby("hk_epsilon")["initial_bridge_active"].mean().to_string(),\n)\nprint(\n    "\\nInitial active-edge fraction by epsilon:\\n",\n    registry_df.groupby("hk_epsilon")["initial_active_edge_fraction"].mean().to_string(),\n)\n'

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    # 1) Identifier parameterization.
    if "from opinion_dynamics.identify_nonlinear import (" in src:
        if VARIANT == "unbounded":
            src = src.replace(
                "from opinion_dynamics.identify_nonlinear import (",
                "from opinion_dynamics.identify_nonlinear_unbounded import (",
                1,
            )
        patched_identifier = True

    # 2) Short isolated cache/result namespace.
    src2, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        f'STUDY_NAME = "{STUDY_NAME_OVERRIDE}"',
        src,
        count=1,
    )
    if n:
        src = src2
        patched_study = True

    src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        f'PIPELINE_VERSION = "{PIPELINE_OVERRIDE}"',
        src,
        count=1,
    )

    old_name = "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed"
    if old_name in src:
        src = src.replace(old_name, STUDY_NAME_OVERRIDE)
        patched_results = True

    # 3) Final simplified controller: M = A_alpha, i.e. lambda_mix = 1.
    src2, n = re.subn(
        r'\bLAMBDA_MIX\s*=\s*0\.70\b',
        'LAMBDA_MIX = 1.0',
        src,
        count=1,
    )
    if n:
        src = src2
        patched_lambda = True

    # 4) Replace the 195-environment registry by the HK epsilon sweep.
    if (
        "ENVIRONMENT_SPECS: list[dict[str, Any]] = []" in src
        and "assert len(ENVIRONMENT_SPECS) == 195" in src
    ):
        src = CUSTOM_ENV_CELL
        patched_env_registry = True

    # 5) Preserve sweep metadata in every output row.
    if "def common_row(spec: dict[str, Any])" in src:
        marker = '        "environment_seed": spec.get("environment_seed"),\n'
        addition = (
            marker
            + '        "hk_epsilon": float(spec["dynamics_params"]["hk_epsilon"]),\n'
            + '        "initial_active_edge_fraction": float(spec.get("initial_active_edge_fraction", np.nan)),\n'
            + '        "initial_bridge_opinion_gap": float(spec.get("initial_bridge_opinion_gap", np.nan)),\n'
            + '        "initial_bridge_active": bool(spec.get("initial_bridge_active", False)),\n'
            + '        "alpha_variant": VARIANT,\n'
        )
        if marker not in src:
            raise RuntimeError("Could not locate common_row environment_seed field.")
        src = src.replace(marker, addition, 1)
        patched_common_row = True

    # 6) Make the configuration hash explicitly encode this diagnostic.
    if "SCIENTIFIC_CONFIG = {" in src:
        marker = "SCIENTIFIC_CONFIG = {\n"
        alpha_name = (
            "sigmoid_bounded_0_1"
            if VARIANT == "bounded"
            else "softplus_over_log2_positive_unbounded"
        )
        extra = (
            marker
            + f'    "alpha_parameterization": "{alpha_name}",\n'
            + '    "diagnostic_scope": "structured_hk_epsilon_sweep",\n'
            + '    "hk_epsilon_values": HK_EPS_VALUES,\n'
        )
        src = src.replace(marker, extra, 1)
        patched_config = True

    patched_cells.append((cell_index, src))

required = {
    "identifier variant": patched_identifier,
    "short STUDY_NAME": patched_study,
    "short results namespace": patched_results,
    "lambda_mix=1.0": patched_lambda,
    "HK epsilon environment registry": patched_env_registry,
    "sweep metadata in output rows": patched_common_row,
    "diagnostic scientific config": patched_config,
}
missing = [name for name, ok in required.items() if not ok]
if missing:
    raise RuntimeError("Expected benchmark structure missing: " + ", ".join(missing))

for name in required:
    print("  OK:", name)

worker_mode = os.environ.get("ABUNDANT_FULL_SHARD_ID") not in (None, "")
print("Driver mode:", "worker" if worker_mode else "final validation/merge")

# Preserve the race-condition fix used by the successful abundant alpha run:
# workers execute through the training cell, while only the final non-worker
# invocation executes the global validation/merge cell.
cells_to_run = patched_cells[:-1] if worker_mode else patched_cells
if worker_mode:
    print("Worker mode: skipping base notebook final global validation/merge cell.")

g = globals()
for cell_index, src in cells_to_run:
    print(f"[base code cell {cell_index}]")
    exec(compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"), g, g)
